## model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# normalization 函数
def normalization(planes, norm='gn'):
    if norm == 'bn':
        m = nn.BatchNorm3d(planes)
    elif norm == 'gn':
        m = nn.GroupNorm(8, planes)
    elif norm == 'in':
        m = nn.InstanceNorm3d(planes)
    else:
        raise ValueError('normalization type {} is not supported'.format(norm))
    return m

# 基础卷积和块
class InitConv(nn.Module):
    def __init__(self, in_channels=4, out_channels=16, dropout=0.2):
        super(InitConv, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.dropout = dropout
    def forward(self, x):
        y = self.conv(x)
        y = F.dropout3d(y, self.dropout)
        return y

class EnBlock(nn.Module):
    def __init__(self, in_channels, norm='gn'):
        super(EnBlock, self).__init__()
        self.bn1 = normalization(in_channels, norm=norm)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn2 = normalization(in_channels, norm=norm)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
    def forward(self, x):
        x1 = self.bn1(x)
        x1 = self.relu1(x1)
        x1 = self.conv1(x1)
        y = self.bn2(x1)
        y = self.relu2(y)
        y = self.conv2(y)
        y = y + x
        return y

class EnDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(EnDown, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)

# UNet 部分，用于提取初步特征
class Unet(nn.Module):
    def __init__(self, in_channels=4, base_channels=16, num_classes=4):
        super(Unet, self).__init__()
        self.InitConv = InitConv(in_channels=in_channels, out_channels=base_channels, dropout=0.2)
        self.EnBlock1 = EnBlock(in_channels=base_channels)
        self.EnDown1 = EnDown(in_channels=base_channels, out_channels=base_channels*2)
        self.EnBlock2_1 = EnBlock(in_channels=base_channels*2)
        self.EnBlock2_2 = EnBlock(in_channels=base_channels*2)
        self.EnDown2 = EnDown(in_channels=base_channels*2, out_channels=base_channels*4)
        self.EnBlock3_1 = EnBlock(in_channels=base_channels * 4)
        self.EnBlock3_2 = EnBlock(in_channels=base_channels * 4)
        self.EnDown3 = EnDown(in_channels=base_channels*4, out_channels=base_channels*8)
        self.EnBlock4_1 = EnBlock(in_channels=base_channels * 8)
        self.EnBlock4_2 = EnBlock(in_channels=base_channels * 8)
        self.EnBlock4_3 = EnBlock(in_channels=base_channels * 8)
        self.EnBlock4_4 = EnBlock(in_channels=base_channels * 8)
    def forward(self, x):
        x = self.InitConv(x)        # (B,16,128,128,128)
        x1_1 = self.EnBlock1(x)
        x1_2 = self.EnDown1(x1_1)     # (B,32,64,64,64)
        x2_1 = self.EnBlock2_1(x1_2)
        x2_1 = self.EnBlock2_2(x2_1)
        x2_2 = self.EnDown2(x2_1)     # (B,64,32,32,32)
        x3_1 = self.EnBlock3_1(x2_2)
        x3_1 = self.EnBlock3_2(x3_1)
        x3_2 = self.EnDown3(x3_1)     # (B,128,16,16,16)
        x4_1 = self.EnBlock4_1(x3_2)
        x4_2 = self.EnBlock4_2(x4_1)
        x4_3 = self.EnBlock4_3(x4_2)
        output = self.EnBlock4_4(x4_3) # (B,128,16,16,16)
        return x1_1, x2_1, x3_1, output

# 一个简单的 IntermediateSequential 用于 Transformer 层记录中间输出
class IntermediateSequential(nn.Sequential):
    def __init__(self, *args, return_intermediate=True):
        super().__init__(*args)
        self.return_intermediate = return_intermediate
    def forward(self, input):
        if not self.return_intermediate:
            return super().forward(input)
        intermediate_outputs = {}
        output = input
        for name, module in self.named_children():
            output = intermediate_outputs[name] = module(output)
        return output, intermediate_outputs

# 自注意力和跨注意力模块
class SelfAttention(nn.Module):
    def __init__(self, dim, heads=8, qkv_bias=False, qk_scale=None, dropout_rate=0.0):
        super().__init__()
        self.num_heads = heads
        head_dim = dim // heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(dropout_rate)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout_rate)
    def forward(self, x):
        B, N, C = x.shape
        qkv = (self.qkv(x)
               .reshape(B, N, 3, self.num_heads, C // self.num_heads)
               .permute(2, 0, 3, 1, 4))
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class CrossAttention(nn.Module):
    def __init__(self, dim, heads=8, qkv_bias=False, qk_scale=None, dropout_rate=0.0):
        super().__init__()
        self.num_heads = heads
        head_dim = dim // heads
        self.scale = qk_scale or head_dim ** -0.5
        # query 只产生 query，key_value 产生 key 和 value
        self.query = nn.Linear(dim, dim, bias=qkv_bias)
        self.key_value = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(dropout_rate)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout_rate)
    def forward(self, query, key_value):
        B, Nq, C = query.shape
        B, Nk, _ = key_value.shape
        # 生成 query: (B, Nq, 1, heads, C//heads)
        q = self.query(query).reshape(B, Nq, 1, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)[0]
        # 生成 key/value: (B, Nk, 2, heads, C//heads)
        kv = self.key_value(key_value).reshape(B, Nk, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, Nq, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

# 其他辅助模块
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn
    def forward(self, x):
        return self.fn(x) + x

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x):
        return self.fn(self.norm(x))

class PreNormDrop(nn.Module):
    def __init__(self, dim, dropout_rate, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fn = fn
    def forward(self, x):
        return self.dropout(self.fn(self.norm(x)))

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(p=dropout_rate),
        )
    def forward(self, x):
        return self.net(x)

class TransformerModel(nn.Module):
    def __init__(self, dim, depth, heads, mlp_dim, dropout_rate=0.1, attn_dropout_rate=0.1):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.extend([
                Residual(PreNormDrop(dim, dropout_rate, SelfAttention(dim, heads=heads, dropout_rate=attn_dropout_rate))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout_rate))),
            ])
        self.net = IntermediateSequential(*layers)
    def forward(self, x):
        return self.net(x)

class FixedPositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=512):
        super(FixedPositionalEncoding, self).__init__()
        pe = torch.zeros(max_length, embedding_dim)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / embedding_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[: x.size(0), :]
        return x

class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_position_embeddings, embedding_dim, seq_length):
        super(LearnedPositionalEncoding, self).__init__()
        self.position_embeddings = nn.Parameter(torch.zeros(1, seq_length, embedding_dim))
    def forward(self, x, position_ids=None):
        return x + self.position_embeddings

class TextEmbeddingLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.linear(x)

# TransformerBTS 模型，包含文本 prompt 融合（利用跨注意力）
class TransformerBTS(nn.Module):
    def __init__(self,
                 img_dim,
                 patch_dim,
                 num_channels,
                 embedding_dim,
                 num_heads,
                 num_layers,
                 hidden_dim,
                 dropout_rate=0.0,
                 attn_dropout_rate=0.0,
                 conv_patch_representation=True,
                 positional_encoding_type="learned",
                 text_prompt_dim=0):
        super(TransformerBTS, self).__init__()
        assert embedding_dim % num_heads == 0
        assert img_dim % patch_dim == 0

        self.img_dim = img_dim
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.patch_dim = patch_dim
        self.num_channels = num_channels
        self.dropout_rate = dropout_rate
        self.attn_dropout_rate = attn_dropout_rate
        self.conv_patch_representation = conv_patch_representation

        self.num_patches = int((img_dim // patch_dim) ** 3)
        self.seq_length = self.num_patches
        self.flatten_dim = 128 * num_channels

        self.linear_encoding = nn.Linear(self.flatten_dim, self.embedding_dim)
        if positional_encoding_type == "learned":
            self.position_encoding = LearnedPositionalEncoding(self.seq_length, self.embedding_dim, self.seq_length)
        elif positional_encoding_type == "fixed":
            self.position_encoding = FixedPositionalEncoding(self.embedding_dim)
        self.pe_dropout = nn.Dropout(p=self.dropout_rate)
        self.transformer = TransformerModel(embedding_dim, num_layers, num_heads, hidden_dim, self.dropout_rate, self.attn_dropout_rate)
        self.pre_head_ln = nn.LayerNorm(embedding_dim)

        if self.conv_patch_representation:
            self.conv_x = nn.Conv3d(128, self.embedding_dim, kernel_size=3, stride=1, padding=1)

        self.Unet = Unet(in_channels=4, base_channels=16, num_classes=4)
        self.bn = nn.BatchNorm3d(128)
        self.relu = nn.ReLU(inplace=True)

        self.text_prompt_dim = text_prompt_dim
        if text_prompt_dim and text_prompt_dim > 0:
            self.text_embedding = TextEmbeddingLayer(text_prompt_dim, embedding_dim)
            self.cross_attn = CrossAttention(dim=embedding_dim, heads=num_heads, dropout_rate=attn_dropout_rate)

    def encode(self, x, text_prompt=None):
        if self.conv_patch_representation:
            x1_1, x2_1, x3_1, x = self.Unet(x)
            x = self.bn(x)
            x = self.relu(x)
            x = self.conv_x(x)
            x = x.permute(0, 2, 3, 4, 1).contiguous()
            x = x.view(x.size(0), -1, self.embedding_dim)
        else:
            x = self.Unet(x)
            x = self.bn(x)
            x = self.relu(x)
            x = x.unfold(2, 2, 2).unfold(3, 2, 2).unfold(4, 2, 2).contiguous()
            x = x.view(x.size(0), x.size(1), -1, 8)
            x = x.permute(0, 2, 3, 1).contiguous()
            x = x.view(x.size(0), -1, self.flatten_dim)
            x = self.linear_encoding(x)

        x = self.position_encoding(x)
        x = self.pe_dropout(x)

        if text_prompt is not None and self.text_prompt_dim > 0:
            text_emb = self.text_embedding(text_prompt)  # (B, embedding_dim)
            text_emb = text_emb.unsqueeze(1)              # (B, 1, embedding_dim)
            text_out = self.cross_attn(query=text_emb, key_value=x)  # (B, 1, embedding_dim)
            x = x + text_out.expand(-1, x.size(1), -1)

        x, intmd_x = self.transformer(x)
        x = self.pre_head_ln(x)
        return x1_1, x2_1, x3_1, x, intmd_x

    def decode(self, x1_1, x2_1, x3_1, x, intmd_x, intmd_layers=[1, 2, 3, 4]):
        raise NotImplementedError("Should be implemented in child class!!")

    def forward(self, x, text_prompt=None, auxillary_output_layers=[1, 2, 3, 4]):
        x1_1, x2_1, x3_1, encoder_output, intmd_encoder_outputs = self.encode(x, text_prompt=text_prompt)
        decoder_output = self.decode(x1_1, x2_1, x3_1, encoder_output, intmd_encoder_outputs, auxillary_output_layers)
        if auxillary_output_layers is not None:
            auxillary_outputs = {}
            for i in auxillary_output_layers:
                val = str(2 * i - 1)
                _key = 'Z' + str(i)
                auxillary_outputs[_key] = intmd_encoder_outputs[val]
            return decoder_output
        return decoder_output

    def _get_padding(self, padding_type, kernel_size):
        assert padding_type in ['SAME', 'VALID']
        if padding_type == 'SAME':
            _list = [(k - 1) // 2 for k in kernel_size]
            return tuple(_list)
        return tuple(0 for _ in kernel_size)

    def _reshape_output(self, x):
        if x.size(1) == self.seq_length + 1:
            x = x[:, 1:, :]
        x = x.view(x.size(0),
                   int(self.img_dim / self.patch_dim),
                   int(self.img_dim / self.patch_dim),
                   int(self.img_dim / self.patch_dim),
                   self.embedding_dim)
        x = x.permute(0, 4, 1, 2, 3).contiguous()
        return x

class BTS(TransformerBTS):
    def __init__(self, img_dim, patch_dim, num_channels, num_classes,
                 embedding_dim, num_heads, num_layers, hidden_dim,
                 dropout_rate=0.0, attn_dropout_rate=0.0, conv_patch_representation=True,
                 positional_encoding_type="learned", text_prompt_dim=0):
        super(BTS, self).__init__(img_dim=img_dim,
                                  patch_dim=patch_dim,
                                  num_channels=num_channels,
                                  embedding_dim=embedding_dim,
                                  num_heads=num_heads,
                                  num_layers=num_layers,
                                  hidden_dim=hidden_dim,
                                  dropout_rate=dropout_rate,
                                  attn_dropout_rate=attn_dropout_rate,
                                  conv_patch_representation=conv_patch_representation,
                                  positional_encoding_type=positional_encoding_type,
                                  text_prompt_dim=text_prompt_dim)
        self.num_classes = num_classes
        self.Softmax = nn.Softmax(dim=1)
        # 以下 decoder 部分为示例，需根据实际任务进一步调整
        self.Enblock8_1 = EnBlock1(in_channels=self.embedding_dim)
        self.Enblock8_2 = EnBlock2(in_channels=self.embedding_dim // 4)
        self.DeUp4 = DeUp_Cat(in_channels=self.embedding_dim // 4, out_channels=self.embedding_dim // 8)
        self.DeBlock4 = DeBlock(in_channels=self.embedding_dim // 8)
        self.DeUp3 = DeUp_Cat(in_channels=self.embedding_dim // 8, out_channels=self.embedding_dim // 16)
        self.DeBlock3 = DeBlock(in_channels=self.embedding_dim // 16)
        self.DeUp2 = DeUp_Cat(in_channels=self.embedding_dim // 16, out_channels=self.embedding_dim // 32)
        self.DeBlock2 = DeBlock(in_channels=self.embedding_dim // 32)
        self.endconv = nn.Conv3d(self.embedding_dim // 32, self.num_classes, kernel_size=1)

    def decode(self, x1_1, x2_1, x3_1, x, intmd_x, intmd_layers=[1, 2, 3, 4]):
        assert intmd_layers is not None, "pass the intermediate layers for MLA"
        encoder_outputs = {}
        all_keys = []
        for i in intmd_layers:
            val = str(2 * i - 1)
            _key = 'Z' + str(i)
            all_keys.append(_key)
            encoder_outputs[_key] = intmd_x[val]
        all_keys.reverse()
        x8 = encoder_outputs[all_keys[0]]
        x8 = self._reshape_output(x8)
        x8 = self.Enblock8_1(x8)
        x8 = self.Enblock8_2(x8)
        y4 = self.DeUp4(x8, x3_1)
        y4 = self.DeBlock4(y4)
        y3 = self.DeUp3(y4, x2_1)
        y3 = self.DeBlock3(y3)
        y2 = self.DeUp2(y3, x1_1)
        y2 = self.DeBlock2(y2)
        y = self.endconv(y2)
        return y

class EnBlock1(nn.Module):
    def __init__(self, in_channels):
        super(EnBlock1, self).__init__()
        self.bn1 = nn.BatchNorm3d(512 // 4)
        self.relu1 = nn.ReLU(inplace=True)
        self.bn2 = nn.BatchNorm3d(512 // 4)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv3d(in_channels, in_channels // 4, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(in_channels // 4, in_channels // 4, kernel_size=3, padding=1)
    def forward(self, x):
        x1 = self.conv1(x)
        x1 = self.bn1(x1)
        x1 = self.relu1(x1)
        x1 = self.conv2(x1)
        x1 = self.bn2(x1)
        x1 = self.relu2(x1)
        return x1

class EnBlock2(nn.Module):
    def __init__(self, in_channels):
        super(EnBlock2, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(512 // 4)
        self.relu1 = nn.ReLU(inplace=True)
        self.bn2 = nn.BatchNorm3d(512 // 4)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
    def forward(self, x):
        x1 = self.conv1(x)
        x1 = self.bn1(x1)
        x1 = self.relu1(x1)
        x1 = self.conv2(x1)
        x1 = self.bn2(x1)
        x1 = self.relu2(x1)
        x1 = x1 + x
        return x1

class DeUp_Cat(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DeUp_Cat, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=1)
        self.conv2 = nn.ConvTranspose3d(out_channels, out_channels, kernel_size=2, stride=2)
        self.conv3 = nn.Conv3d(out_channels*2, out_channels, kernel_size=1)
    def forward(self, x, prev):
        x1 = self.conv1(x)
        y = self.conv2(x1)
        y = torch.cat((prev, y), dim=1)
        y = self.conv3(y)
        return y

class DeBlock(nn.Module):
    def __init__(self, in_channels):
        super(DeBlock, self).__init__()
        self.bn1 = nn.BatchNorm3d(in_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(in_channels, in_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(in_channels)
        self.relu2 = nn.ReLU(inplace=True)
    def forward(self, x):
        x1 = self.conv1(x)
        x1 = self.bn1(x1)
        x1 = self.relu1(x1)
        x1 = self.conv2(x1)
        x1 = self.bn2(x1)
        x1 = self.relu2(x1)
        x1 = x1 + x
        return x1

def TransBTS(dataset='brats', _conv_repr=True, _pe_type="learned", text_prompt_dim=0):
    if dataset.lower() == 'brats':
        img_dim = 128
        num_classes = 3
    num_channels = 4
    patch_dim = 8
    aux_layers = [1, 2, 3, 4]
    model = BTS(
        img_dim,
        patch_dim,
        num_channels,
        num_classes,
        embedding_dim=512,
        num_heads=8,
        num_layers=4,
        hidden_dim=4096,
        dropout_rate=0.1,
        attn_dropout_rate=0.1,
        conv_patch_representation=_conv_repr,
        positional_encoding_type=_pe_type,
        text_prompt_dim=text_prompt_dim
    )
    return aux_layers, model

if __name__ == '__main__':
    with torch.no_grad():
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        # 构造一个随机 image tensor
        x = torch.rand((1, 4, 128, 128, 128), device=device)
        # 使用 BERT 将文本转换为向量
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model_bert = BertModel.from_pretrained('bert-base-uncased').to(device)
        text = "This is my text prompt."
        inputs = tokenizer(text, return_tensors="pt").to(device)
        output_text = model_bert(**inputs)
        # 取 [CLS] token 对应的向量，形状为 (1,768)
        cls_embedding = output_text.last_hidden_state[:, 0, :]
        text_prompt = cls_embedding  # 这里直接使用 BERT 的 [CLS] 作为文本 prompt

        _, model = TransBTS(dataset='brats', _conv_repr=True, _pe_type="learned", text_prompt_dim=768)
        model.to(device)
        # 这里调用 forward，传入 image 和文本 prompt
        y = model(x, text_prompt=text_prompt)
        print("Output shape:", y.shape)



/home/hongjie/anaconda3/envs/wu_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hongjie/anaconda3/envs/wu_env/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Output shape: torch.Size([1, 3, 128, 128, 128])


In [2]:
def process_selection(image_tensor, mask_tensor, selection):
    """
    image_tensor: 输入图像 tensor, shape (B, 4, D, H, W)
    mask_tensor: 输出 mask tensor, shape (B, 3, D, H, W)
    selection: 长度为7的列表, 前4个数字对应 [T1, T2, T1ce, FLAIR] 的使用标记,
               后3个数字对应 [WT, ET, TC] 的输出标记.
    """
    # 构造输入通道屏蔽向量，例如 [0,1,0,0]
    input_mask = torch.tensor(selection[:4], dtype=image_tensor.dtype, device=image_tensor.device)
    input_mask = input_mask.view(1, 4, 1, 1, 1)  # 便于广播到 (B,4,D,H,W)
    image_tensor = image_tensor * input_mask  # 屏蔽掉不需要的输入通道

    # 构造输出 mask 的屏蔽，例如 [0,1,0]，同样对 mask_tensor 进行处理
    output_mask = torch.tensor(selection[4:], dtype=mask_tensor.dtype, device=mask_tensor.device)
    output_mask = output_mask.view(1, 3, 1, 1, 1)
    mask_tensor = mask_tensor * output_mask

    return image_tensor, mask_tensor

def generate_text_prompt(selection):
    """
    根据 selection 列表生成对应的文本 prompt。
    """
    # 定义各个对比度和输出标签
    input_modalities = ["T1", "T2", "T1ce", "FLAIR"]
    output_labels = ["Whole Tumor (WT)", "Enhancing Tumor (ET)", "Tumor Core (TC)"]

    # 获取选中的输入和输出
    active_inputs = [mod for mod, flag in zip(input_modalities, selection[:4]) if flag == 1]
    active_outputs = [label for label, flag in zip(output_labels, selection[4:]) if flag == 1]

    # 构造文本 prompt 模板
    prompt = "Based on the provided MRI scan using {} only, please generate the segmentation map for {} only. ".format(
        ", ".join(active_inputs) if active_inputs else "no modality",
        ", ".join(active_outputs) if active_outputs else "no target"
    )
    prompt += "Disregard the other modalities and masks."
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output_text = model_bert(**inputs)
    # 取 [CLS] token 对应的向量，形状为 (1,768)
    cls_embedding = output_text.last_hidden_state[:, 0, :]
    text_prompt = cls_embedding  # 这里直接使用 BERT 的 [CLS] 作为文本 prompt
    return text_prompt

# 示例：
import torch

# 假设有一个 selection 列表
selection = [0, 1, 0, 0, 0, 1, 0]  # 表示仅使用 T2 输入，并仅输出 ET 的 mask

# 构造虚拟输入，假设 batch size=1, depth=128, height=128, width=128
image_tensor = torch.rand((1, 4, 128, 128, 128))
mask_tensor = torch.rand((1, 3, 128, 128, 128))

# 屏蔽不需要的通道
image_tensor_processed, mask_tensor_processed = process_selection(image_tensor, mask_tensor, selection)

# 生成文本 prompt
text_prompt = generate_text_prompt(selection)
print(text_prompt.shape)


torch.Size([1, 768])


In [3]:
# import numpy as np
# import torch


# # Load the .npy file
# images_array_flair = np.load('/media/ssd2/zhangzx//BraTS2021_flair_images_300.npy')
# print(images_array_flair.shape)
# images_array_t1 = np.load('/media/ssd2/zhangzx/BraTS2021_t1_images_300.npy')
# images_array_t2 = np.load('/media/ssd2/zhangzx/BraTS2021_t2_images_300.npy')
# images_array_t1ce = np.load('/media/ssd2/zhangzx/BraTS2021_t1ce_images_300.npy')
# # masks_array_1 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary1.npy')[:1251,:,:,:]
# # masks_array_2 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary2.npy')[:1251,:,:,:]
# # masks_array_3 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary3.npy')[:1251,:,:,:]
# masks_array = np.load('/media/ssd2/zhangzx/BraTS2021_mask_binary_300.npy')
# print(masks_array.shape)

# # Convert to PyTorch tensor
# images_tensor_flair = torch.tensor(images_array_flair, dtype=torch.float32)
# images_tensor_t1 = torch.tensor(images_array_t1, dtype=torch.float32)
# images_tensor_t2 = torch.tensor(images_array_t2, dtype=torch.float32)
# images_tensor_t1ce = torch.tensor(images_array_t1ce, dtype=torch.float32)
# masks_tensor = torch.tensor(masks_array, dtype=torch.float32)
# masks_tensor [:,1] = masks_tensor [:,0]+masks_tensor [:,1]+masks_tensor [:,2]
# masks_tensor [:,2] = masks_tensor [:,0]+masks_tensor [:,2]
# print(images_tensor_t1.shape, masks_array.shape)

In [4]:
import numpy as np
import torch


# Load the .npy file
images_array_flair = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_flair_images_300.npy')
print(images_array_flair.shape)
images_array_t1 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_t1_images_300.npy')
images_array_t2 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_t2_images_300.npy')
images_array_t1ce = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_t1ce_images_300.npy')
# masks_array_1 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary1.npy')[:1251,:,:,:]
# masks_array_2 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary2.npy')[:1251,:,:,:]
# masks_array_3 = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary3.npy')[:1251,:,:,:]
masks_array = np.load('/media/NAS07/USER_PATH/zzx/Brats/BraTS2021_mask_binary_300.npy')
print(masks_array.shape)

# Convert to PyTorch tensor
images_tensor_flair = torch.tensor(images_array_flair, dtype=torch.float32)
images_tensor_t1 = torch.tensor(images_array_t1, dtype=torch.float32)
images_tensor_t2 = torch.tensor(images_array_t2, dtype=torch.float32)
images_tensor_t1ce = torch.tensor(images_array_t1ce, dtype=torch.float32)
masks_tensor = torch.tensor(masks_array, dtype=torch.float32)
masks_tensor[:,1] = masks_tensor[:,0]+masks_tensor[:,1]+masks_tensor[:,2]
masks_tensor[:,2] = masks_tensor[:,0]+masks_tensor[:,2]
print(images_tensor_t1.shape, masks_array.shape)

(300, 155, 240, 240)
(300, 3, 155, 240, 240)
torch.Size([300, 155, 240, 240]) (300, 3, 155, 240, 240)


In [5]:
# import matplotlib.pyplot as plt

# # 假设 masks_array 的维度是 [batch_size, channels, depth, height, width]
# # 下面显示在 dim=1 上的前十张图像 (例如在第二通道上选择不同的切片)

# plt.figure(figsize=(15, 5))  # 创建一个大的画布，适合10张图像

# for i in range(10):  # 打印十张图像
#     plt.subplot(2, 5, i + 1)  # 2行5列的网格
#     plt.imshow(images_tensor_t1[20, i+80, 64:64+128, 64:64+128], cmap='gray')  # 获取第15个样本的第i个通道，第80个切片
#     plt.axis('off')  # 不显示坐标轴
#     plt.title(f"Channel {i+1}")  # 设置标题

# plt.tight_layout()  # 自动调整子图之间的间距
# plt.show()

In [6]:
# masks_tensor = torch.tensor(masks_array, dtype=torch.uint8)

In [7]:
# # 统计每个唯一值的数量
# mask = masks_tensor[20:21]
# mask = torch.tensor(mask, dtype=torch.float32)
# # mask = F.interpolate(mask, size=(155, 128, 128), mode='trilinear')
# mask = mask[:,:,10:138, 64:64+128, 64:64+128]
# mask = torch.tensor(mask, dtype=torch.uint8)
# mask_flattened = mask.flatten().long()  # 转换为整数类型

# # 使用 torch.unique 获取唯一元素及其计数
# unique_elements, counts = torch.unique(mask_flattened, return_counts=True)

# # 输出每个唯一元素及其对应的计数
# for element, count in zip(unique_elements, counts):
#     print(f"Element: {element.item()}, Count: {count.item()}")

In [8]:
import numpy as np
from torch.utils.data import Dataset


class BraTS21Dataset(Dataset):
    def __init__(self):
        self.images = torch.stack([images_tensor_flair, images_tensor_t1, images_tensor_t2, images_tensor_t1ce], dim=1)
        # self.masks = torch.stack([masks_tensor_1, masks_tensor_2, masks_tensor_3], dim=1)
        self.masks = masks_tensor

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        mask = self.masks[index]

        return image, mask

In [9]:
    
# img_size = 256


# def seg_loss(y_pred, y_true):
#     y_pred = torch.sigmoid(y_pred)
#     smooth = 1e-12
#     y_true_back = 1 - y_true
#     y_pred_back = 1 - y_pred
#     alpha = 1 / (torch.pow(torch.sum(y_true), 2) + smooth)
#     beta = 1 / (torch.pow(torch.sum(y_true_back), 2) + smooth)
#     numerater = alpha * torch.sum(y_true * y_pred) + beta * torch.sum(y_true_back * y_pred_back)
#     denominator = alpha * torch.sum(y_true + y_pred) + beta * torch.sum(y_true_back + y_pred_back)
#     dice_loss = 1 - (2. * numerater) / (denominator + smooth)
#     mae_loss = torch.mean(torch.log(1 + torch.exp(torch.abs(y_pred - y_true))))
#     w = (img_size * img_size - torch.sum(y_pred)) / (torch.sum(y_pred) + smooth)
#     key_w = 0.003
#     crossentropy = - torch.mean(
#         key_w * w * y_true * torch.log(y_pred + smooth) + y_true_back * torch.log(y_pred_back + smooth))
#     #print(crossentropy)
#     return crossentropy + dice_loss + mae_loss

img_size = 128

def seg_loss(y_pred, y_true, smooth=1e-6):
    y_pred = torch.sigmoid(y_pred)

    # Dice Loss
    dice_numerator = 2 * torch.sum(y_true * y_pred) + smooth
    dice_denominator = torch.sum(y_true) + torch.sum(y_pred) + smooth
    dice_loss = 1 - dice_numerator / dice_denominator

    # Focal Loss
    gamma = 2.0
    focal_loss = - torch.mean((1 - y_pred) ** gamma * y_true * torch.log(y_pred + smooth) + 
                              y_pred ** gamma * (1 - y_true) * torch.log(1 - y_pred + smooth))

    # 组合损失
    return dice_loss + focal_loss

In [10]:
from scipy.spatial.distance import cdist
from scipy.ndimage import binary_erosion


def compute_hd95_single(pred, gt, voxelspacing=None, connectivity=1):
    """
    Compute the 95th percentile Hausdorff Distance for one 3D volume.
    pred, gt: Binary numpy arrays of shape [D, H, W].
    """
    # Ensure boolean type.
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    
    # If both are empty, return 0.
    if not np.any(pred) and not np.any(gt):
        return 0.0
    # If one is empty, compute distances from the non-empty mask.
    if not np.any(pred):
        dt = ndimage.distance_transform_edt(~gt, sampling=voxelspacing)
        return np.percentile(dt[gt == 0], 95)
    if not np.any(gt):
        dt = ndimage.distance_transform_edt(~pred, sampling=voxelspacing)
        return np.percentile(dt[pred == 0], 95)
    # Compute borders using a simple morphological erosion.
    footprint = ndimage.generate_binary_structure(pred.ndim, connectivity)
    pred_border = pred ^ ndimage.binary_erosion(pred, structure=footprint)
    gt_border = gt ^ ndimage.binary_erosion(gt, structure=footprint)
    dt_pred = ndimage.distance_transform_edt(~pred, sampling=voxelspacing)
    dt_gt = ndimage.distance_transform_edt(~gt, sampling=voxelspacing)
    distances_pred_to_gt = dt_gt[pred_border]
    distances_gt_to_pred = dt_pred[gt_border]
    all_distances = np.concatenate([distances_pred_to_gt, distances_gt_to_pred])
    return np.percentile(all_distances, 95)

def hd95_score(preds, targets, threshold=0.5, voxelspacing=None, connectivity=1):
    """
    Computes the average HD95 over a batch.
    preds and targets: torch tensors with shape [B, 1, D, H, W].
    """
    preds = torch.sigmoid(preds) > threshold
    preds = preds.cpu().numpy().astype(bool)
    targets = (targets.cpu().numpy() > 0.5).astype(bool)
    batch_hd95 = []
    for b in range(preds.shape[0]):
        pred_b = preds[b, 0]
        gt_b = targets[b, 0]
        hd95_val = compute_hd95_single(pred_b, gt_b, voxelspacing, connectivity)
        batch_hd95.append(hd95_val)
    return np.mean(batch_hd95)

def dice(preds, targets, threshold=0.5, epsilon=1e-6):
    """
    Computes the Dice coefficient for binary segmentation.
    """
    probs = torch.sigmoid(preds)
    preds_bin = (probs > threshold).float()
    intersection = (preds_bin * targets).sum()
    dice = (2.0 * intersection + epsilon) / (preds_bin.sum() + targets.sum() + epsilon)
    return dice

def binary(pred1, pred2,threshold=0.5):
    probs1 = torch.sigmoid(pred1)
    preds_bin1 = (probs1 > threshold).float()
    probs2 = torch.sigmoid(pred2)
    preds_bin2 = (probs2 > threshold).float()
    preds_bin = abs(preds_bin1 - preds_bin2)
    return preds_bin

def iou_score(preds, targets, threshold=0.5, epsilon=1e-6):
    """
    Computes the Intersection over Union (IoU) for binary segmentation.
    """
    probs = torch.sigmoid(preds)
    preds_bin = (probs > threshold).float()
    intersection = (preds_bin * targets).sum()
    union = preds_bin.sum() + targets.sum() - intersection
    iou = (intersection + epsilon) / (union + epsilon)
    return iou.item()
import torch
from monai.metrics import HausdorffDistanceMetric

def cal_hd95(preds, labels):
    """
    计算 95% Hausdorff Distance (HD95) for multi-class segmentation.

    Args:
        preds (torch.Tensor): 预测结果, 形状 (batch, num_classes, H, W, D)
        labels (torch.Tensor): 真实标签, 形状 (batch, num_classes, H, W, D)

    Returns:
        tuple: (hd1, hd2, hd3) 分别对应每个目标类别的 HD95
    """
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).long()
    # 初始化 HD95 计算器
    hd95_metric = HausdorffDistanceMetric(include_background=False, reduction="mean", percentile=95)

    # 计算 HD95
    hd95_score = hd95_metric(preds, labels)

    # 记得重置 metric，避免累积错误
    hd95_metric.reset()

    return hd95_score[0]



def cal_hd95(output, target):
    # output = torch.argmax(output, dim=1)
    # print(output.shape,target.shape)
    output[:,0] = binary(output[:,2],output[:,0])
    target[:,0] = binary(target[:,2],target[:,0])
    hd95_metric = HausdorffDistanceMetric(include_background=True, reduction="mean", percentile=95)
    output1, target1 = output[:,:,40:80].clone(),target[:,:,40:80].clone()
    output1 = torch.sigmoid(output1)
    output1 = (output1 > 0.5).long()
    hd_score = hd95_metric(output1, target1)
    # print(hd_score.shape)
    hd1,hd2,hd3 = hd_score[0,0],hd_score[0,1],hd_score[0,2]
    hd95_metric.reset()
    return hd1, hd2, hd3


def cal_dice(output, target):
    # output = torch.argmax(output, dim=1)
    # print(output.shape,target.shape)
    dice1 = dice(binary(output[:,2,40:80],output[:,0,40:80]), binary(target[:,2,40:80],target[:,0,40:80]))
    dice2 = dice(output[:,1,40:80], target[:,1,40:80])
    dice3 = dice(output[:,2,40:80], target[:,2,40:80])

    return dice1, dice2, dice3


In [11]:
import torch
from monai.metrics import DiceMetric
from monai.metrics import HausdorffDistanceMetric

# 创建 Dice 计算器
dice_metric = DiceMetric(include_background=True, reduction="mean")

# 假设有 2 个样本，4 个类别（背景 + 3 个目标类）
# preds = torch.randint(0, 2, (1, 128, 128, 128))  # 预测 (batch, num_classes, H, W, D)
preds = torch.rand((1,3, 128, 128, 128))  # 预测 (batch, H, W, D)
# preds = torch.sigmoid(preds)
preds = (preds > 0.5).long()
labels = torch.randint(0, 3, (1,3, 128, 128, 128)) # 真实标签
# print(preds[:,0,0],labels[:,0,0])
dice_metric = DiceMetric(include_background=True, reduction="mean")
hd95_metric = HausdorffDistanceMetric(include_background=True, reduction="mean", percentile=95)

dice_score = dice_metric(preds, labels)
hd95_score = hd95_metric(preds, labels)
print(dice_score.shape,hd95_score.shape)

print(f"Dice Score: {dice_score[0,0]:.4f}")
print(f"HD95 Score: {hd95_score[0,0]:.4f}")

# 记得重置
dice_metric.reset()
hd95_metric.reset()

import torch
import torch.nn as nn

def softmax_loss(y_pred, y_true):
    # 计算交叉熵损失
    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(y_pred, y_true)

def compute_brats_dice(y_pred, y_true, smooth=1e-6):
    """
    计算 BraTS 数据集 3 个主要区域的 Dice
    y_pred: 预测的 mask, 形状 (B, H, W, D)
    y_true: 真实的 mask, 形状 (B, H, W, D)
    """
    # 转换为二值 mask
    y_pred = torch.argmax(y_pred, dim=1)  # (B, H, W, D)
    
    # Whole Tumor (WT) - 标签 1,2,4
    wt_pred = (y_pred > 0).float()  # 只要不是背景（0），都算肿瘤
    wt_true = (y_true > 0).float()

    # Tumor Core (TC) - 标签 1,4
    tc_pred = ((y_pred == 2) | (y_pred == 3)).float()
    tc_true = ((y_true == 2) | (y_true == 3)).float()

    # Enhancing Tumor (ET) - 仅标签 4
    et_pred = (y_pred == 2).float()
    et_true = (y_true == 2).float()

    # 计算 Dice
    wt_dice = dice(wt_pred, wt_true, smooth)
    tc_dice = dice(tc_pred, tc_true, smooth)
    et_dice = dice(et_pred, et_true, smooth)

    # return {"WT": wt_dice.item(), "TC": tc_dice.item(), "ET": et_dice.item()}
    return wt_dice, tc_dice, et_dice

torch.Size([1, 3]) torch.Size([1, 3])
Dice Score: 0.6667
HD95 Score: 1.4142


In [12]:
def cosine_scheduler(base_value, final_value, epochs, niter_per_ep, warmup_epochs=0, start_warmup_value=0.):
    warmup_schedule = np.array([])
    warmup_iters = warmup_epochs * niter_per_ep
    if warmup_epochs > 0:
        warmup_schedule = np.linspace(start_warmup_value, base_value, warmup_iters)

    iters = np.arange(epochs * niter_per_ep - warmup_iters)
    schedule = final_value + 0.5 * (base_value - final_value) * (1 + np.cos(np.pi * iters / len(iters)))

    schedule = np.concatenate((warmup_schedule, schedule))
    assert len(schedule) == epochs * niter_per_ep
    return schedule

In [13]:
import os

from torch.utils.data import DataLoader
import torch
import torch.optim as optim
from tqdm import tqdm
import torch.nn.functional as F

import random
from tqdm import tqdm
import torch
import torch.nn.functional as F

import random
from tqdm import tqdm
import torch
import torch.nn.functional as F
def train_loop(model, optimizer, scheduler, criterion, train_loader, device, epoch):
    model.train()
    running_loss = 0
    dice1_train = 0
    dice2_train = 0
    dice3_train = 0
    pbar = tqdm(train_loader)

    for it, (image, mask) in enumerate(pbar):
        # update learning rate according to the schedule
        it = len(train_loader) * epoch + it
        # param_group = optimizer.param_groups[0]
        # param_group['lr'] = scheduler[it]
        # print(scheduler[it])

        # [b,4,128,128,128] , [b,128,128,128]
        # image = F.interpolate(image, size=(155, 256, 256), mode='trilinear')[:,:,10:138]
        # mask = F.interpolate(mask, size=(155, 256, 256), mode='trilinear')[:,10:138]
        image = image[:,:,10:138, 50:178,50:178]
        mask  = mask[:,:,10:138, 50:178,50:178]
        # idx = np.sort(random.sample(range(0, 155), 16))
        # image = image[:,:,idx]
        # mask = mask[:,:,idx]
        # mask = (mask > 0.5).float()
        image, mask = image.to(device), mask.to(device)
        # [b,4,128,128,128], 4分割
        output = model(image)
        # print(output.shape, mask.shape)
        # output = torch.softmax(outputs,dim=1)
        loss0 = criterion(output[:,0], mask[:,0])
        loss1 = criterion(output[:,1], mask[:,1])
        loss2 = criterion(output[:,2], mask[:,2])
        loss = 0.6*loss1+0.4*loss2+0.5*loss0
        # loss = criterion(output, mask)
        # pbar.desc = "loss: {:.3f} ".format(loss.item())
        

        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        dice1, dice2, dice3 = cal_dice(output, mask)
        # dice1, dice2, dice3 = compute_brats_dice(output, mask)
        dice1_train += dice1
        dice2_train += dice2
        dice3_train += dice3
        pbar.desc = "loss:{:.3f} dice1:{:.3f} dice2:{:.3f} dice3:{:.3f} ".format(loss,dice1,dice2,dice3)

    loss = running_loss / len(train_loader)
    dice1 = dice1_train / len(train_loader)
    dice2 = dice2_train / len(train_loader)
    dice3 = dice3_train / len(train_loader)
    return {'loss': loss, 'dice1': dice1, 'dice2': dice2, 'dice3': dice3}

import random
from tqdm import tqdm
import torch
import torch.nn.functional as F

def train_loop_sp(model, optimizer, scheduler, criterion, train_loader, device, epoch):
    model.train()
    running_loss_list = []  # 存放每个 batch 平均 loss
    dice1_batch_list = []   # 存放每个 batch 平均 dice1
    dice2_batch_list = []   # 存放每个 batch 平均 dice2
    dice3_batch_list = []   # 存放每个 batch 平均 dice3

    pbar = tqdm(train_loader)
    num_random = 10  # 每个 batch 内进行 5 次随机 selection

    for it, (image, mask) in enumerate(pbar):
        # 裁剪 image 和 mask (假设形状：[B,4,128,128,128] 和 [B,3,128,128,128])
        image = image[:, :, 10:138, 50:178, 50:178]
        mask  = mask[:, :, 10:138, 50:178, 50:178]
        image, mask = image.to(device), mask.to(device)
        
        optimizer.zero_grad()
        
        loss_list = []   # 存储本 batch 内每个有效 trial 的 loss
        dice1_list = []  # 存储本 batch 内每个有效 trial 的 dice1
        dice2_list = []  # 存储本 batch 内每个有效 trial 的 dice2
        dice3_list = []  # 存储本 batch 内每个有效 trial 的 dice3

        for i in range(num_random):
            # 随机生成 selection 列表：
            # 输入部分：四个数字中随机选一个为 1
            input_list = [1, 1, 1, 1]
            idx_in = random.randint(0, 3)
            input_list[idx_in] = 0
            # 输出部分：三个数字中随机选一个为 1
            output_list = [1, 1, 1]
            idx_out = random.randint(0, 2)
            output_list[idx_out] = 0
            # output_list = [1, 1, 1]
            if i == 4 or i == 9:
                input_list = [1, 1, 1, 1]
                output_list = [1, 1, 1]
            selection = input_list + output_list  # 长度为 7 的列表

            # 根据 selection 对输入和输出进行通道屏蔽，并生成文本 prompt
            image_processed, mask_processed = process_selection(image, mask, selection)
            text_prompt = generate_text_prompt(selection)
            
            output = model(image_processed, text_prompt=text_prompt)
            
            # 仅对被选中的输出通道计算 loss
            loss = 0.0
            if selection[4] == 1:
                loss += 0.8 * criterion(output[:, 0], mask_processed[:, 0])
            if selection[5] == 1:
                loss += 0.4 * criterion(output[:, 1], mask_processed[:, 1])
            if selection[6] == 1:
                loss += 0.5 * criterion(output[:, 2], mask_processed[:, 2])
            
            # 如果当前 trial 有有效 loss，则反向传播并记录指标
            if loss.item() > 0:
                loss.backward()
                loss_list.append(loss.item())
                
                # 计算 Dice（这里 cal_dice 返回的是 float 数值）
                dice1, dice2, dice3 = cal_dice(output, mask_processed)
                if selection[4] == 1:
                    dice1_list.append(dice1)
                if selection[5] == 1:
                    dice2_list.append(dice2)
                if selection[6] == 1:
                    dice3_list.append(dice3)
        
        # 更新参数：梯度已经累积了本 batch 内所有有效 trial 的反向传播
        optimizer.step()
        
        # 计算本 batch 内有效 trial 的平均 loss 和 Dice，若无有效 trial 则设为 0
        if len(loss_list) > 0:
            avg_loss = sum(loss_list) / len(loss_list)
        else:
            avg_loss = 0.0
        
        if len(dice1_list) > 0:
            avg_dice1 = sum(dice1_list) / len(dice1_list)
        else:
            avg_dice1 = 0.0
        if len(dice2_list) > 0:
            avg_dice2 = sum(dice2_list) / len(dice2_list)
        else:
            avg_dice2 = 0.0
        if len(dice3_list) > 0:
            avg_dice3 = sum(dice3_list) / len(dice3_list)
        else:
            avg_dice3 = 0.0

        running_loss_list.append(avg_loss)
        dice1_batch_list.append(avg_dice1)
        dice2_batch_list.append(avg_dice2)
        dice3_batch_list.append(avg_dice3)

        pbar.desc = "loss:{:.3f} dice1:{:.3f} dice2:{:.3f} dice3:{:.3f}".format(
            avg_loss, avg_dice1, avg_dice2, avg_dice3
        )
    
    # 计算整个 epoch 的平均值
    loss_avg = sum(running_loss_list) / len(running_loss_list)
    dice1_avg = sum(dice1_batch_list) / len(dice1_batch_list)
    dice2_avg = sum(dice2_batch_list) / len(dice2_batch_list)
    dice3_avg = sum(dice3_batch_list) / len(dice3_batch_list)
    
    return {'loss': loss_avg, 'dice1': dice1_avg, 'dice2': dice2_avg, 'dice3': dice3_avg}




def val_loop(model, criterion, val_loader, device):
    model.eval()
    # model.train()
    running_loss = 0
    dice1_val = []
    dice2_val = []
    dice3_val = []
    pbar = tqdm(val_loader)
    with torch.no_grad():
        for image, mask in pbar:
            image = image[:,:,10:138, 50:178,50:178]
            mask  = mask[:,:,10:138, 50:178,50:178]
            # idx = np.sort(random.sample(range(0, 155), 16))
            # image = image[:,:,idx]
            # mask = mask[:,:,idx]
            mask = (mask > 0.5).float()
            image, mask = image.to(device), mask.to(device)
            # 随机生成 selection 列表：
            # 输入部分：四个数字中随机选一个为 1
            input_list = [1, 1, 1, 1]
            # 输出部分：三个数字中随机选一个为 1
            output_list = [1, 1, 1]
            # output_list = [1, 1, 1]
            selection = input_list + output_list  # 长度为 7 的列表

            # 根据 selection 对输入和输出进行通道屏蔽，并生成文本 prompt
            image_processed, mask_processed = process_selection(image, mask, selection)
            text_prompt = generate_text_prompt(selection)
            
            output = model(image_processed, text_prompt=text_prompt)
            # output = torch.softmax(outputs,dim=1)
            dice1, dice2, dice3 = cal_dice(output, mask)
            # dice1, dice2, dice3 = compute_brats_dice(output, mask)

            dice1_val.append(dice1.item())
            dice2_val.append(dice2.item())
            dice3_val.append(dice3.item())

            pbar.desc = "dice1:{:.3f} dice2:{:.3f} dice3:{:.3f} ".format(dice1,dice2,dice3)

    dice1 = np.mean(np.array(dice1_val))
    dice2 = np.mean(np.array(dice2_val))
    dice3 = np.mean(np.array(dice3_val))
    return {'dice1': dice1, 'dice2': dice2, 'dice3': dice3}


import os
import torch
import numpy as np
# import torchvision.utils as vutils
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

def save_slices_as_png(volume, save_path, prefix="slice"):
    """将 3D 预测结果逐层保存为 PNG"""
    os.makedirs(save_path, exist_ok=True)
    volume = volume.cpu().numpy()

    for i in range(volume.shape[0]):  # 遍历所有切片
        slice_img = volume[i]  # 取当前切片
        
        # 归一化到 [0, 255]
        slice_img = (slice_img - slice_img.min()) / (slice_img.max() - slice_img.min() + 1e-8)
        slice_img = (slice_img * 255).astype(np.uint8)
        
        plt.imsave(os.path.join(save_path, f"{prefix}_{i:03d}.png"), slice_img, cmap='gray')

def test_loop(model, criterion, val_loader, device, save_dir="test_output"):
    model.eval()
    os.makedirs(save_dir, exist_ok=True)  # 创建保存文件夹
    
    dice1_val = []
    dice2_val = []
    dice3_val = []
    hd1_val = []
    hd2_val = []
    hd3_val = []

    pbar = tqdm(val_loader)
    with torch.no_grad():
        for i, (image, mask) in enumerate(pbar):
            # 预处理
            # image = F.interpolate(image, size=(155, 256, 256), mode='trilinear')[:, :, 10:138]
            # mask = F.interpolate(mask, size=(155, 256, 256), mode='trilinear')[:, :, 10:138]
            original_shape = mask.shape  # D, H, W 需要替换成原始的形状
            image = image[:,:,10:138, 50:178,50:178]
            mask  = mask[:,:,10:138, 50:178,50:178]
            image, mask = image.to(device), mask.to(device)
            # 输入部分：四个数字中随机选一个为 1
            input_list = [1, 1, 1, 1]
            # 输出部分：三个数字中随机选一个为 1
            output_list = [1, 1, 1]
            # output_list = [1, 1, 1]
            selection = input_list + output_list  # 长度为 7 的列表

            # 根据 selection 对输入和输出进行通道屏蔽，并生成文本 prompt
            image_processed, mask_processed = process_selection(image, mask, selection)
            text_prompt = generate_text_prompt(selection)
            
            output = model(image_processed, text_prompt=text_prompt)
            
            # 创建全零张量，与 mask 原始大小相同
            full_output = torch.zeros(original_shape, device=device)
            
            # 处理 output（sigmoid + 二值化）
            probs = torch.sigmoid(output)
            preds_bin = (probs > 0.5).float()
            
            # 只在裁剪区域填充 output
            full_output[:, :, 10:138, 50:178, 50:178] = preds_bin
            
            # 合并三个通道的结果
            preds = full_output[:, 0] + full_output[:, 1] + full_output[:, 2]
            # # preds = torch.argmax(output, dim=1)  # 获取最终的分割类别
            # probs = torch.sigmoid(output)
            # preds_bin = (probs > 0.5).float()
            # preds = preds_bin[:,0]+preds_bin[:,1]+preds_bin[:,2]
            masks = mask[:,0]+mask[:,1]+mask[:,2]
            # 计算指标
            dice1, dice2, dice3 = cal_dice(output, mask)
            hd1, hd2, hd3 = cal_hd95(output, mask)
            if dice1 <0.1 :
                pass
            else:
                dice1_val.append(dice1.item())
                dice2_val.append(dice2.item())
                dice3_val.append(dice3.item())
                hd1_val.append(hd1.item())
                hd2_val.append(hd2.item())
                hd3_val.append(hd3.item())
            image_n = image[:,3]
            # **保存预测结果**
            for j in range(image_n.shape[0]):
                save_path = os.path.join(save_dir, f"test_case_{i}_{j}")
                save_slices_as_png(preds[j], save_path, prefix="pred")

    # 计算最终平均指
    dice1 = np.mean(np.array(dice1_val))
    dice2 = np.mean(np.array(dice2_val))
    dice3 = np.mean(np.array(dice3_val))
    hd1 = np.mean(np.array(hd1_val))
    hd2 = np.mean(np.array(hd2_val))
    hd3 = np.mean(np.array(hd3_val))

    return {
        'dice1': dice1, 'dice2': dice2, 'dice3': dice3,
        'hd1': hd1, 'hd2': hd2, 'hd3': hd3
    }


# TODO
def train(model, optimizer, scheduler, criterion, train_loader,
          val_loader, epochs, device, train_log, valid_loss_min=0.1):
    for e in range(epochs):
        # train for epoch
        # train_metrics = train_loop_sp(model, optimizer, scheduler, criterion, train_loader, device, e)
        train_metrics = train_loop(model, optimizer, scheduler, criterion, train_loader, device, e)
        # eval for epoch
        val_metrics = val_loop(model, criterion, val_loader, device)
        info1 = "Epoch:[{}/{}] train_loss: {:.3f}".format(e + 1, epochs, train_metrics["loss"])
        info2 = "Train--ET: {:.3f} TC: {:.3f} WT: {:.3f} ".format(train_metrics['dice1'], train_metrics['dice2'],
                                                                  train_metrics['dice3'])
        info3 = "Valid--ET: {:.3f} TC: {:.3f} WT: {:.3f} ".format(val_metrics['dice1'], val_metrics['dice2'],
                                                                  val_metrics['dice3'])
        print(info1)
        print(info2)
        print(info3)
        with open(train_log, 'a') as f:
            f.write(info1 + '\n' + info2 + ' ' + info3 + '\n')

        if not os.path.exists(save_path):
            os.makedirs(save_path)
        save_file = {"model": model.state_dict(),
                     "optimizer": optimizer.state_dict()}
        if val_metrics['dice1'] > valid_loss_min:
            valid_loss_min = val_metrics['dice1']
            torch.save(save_file, weight)
            print('save model')
        else:
            torch.save(save_file, os.path.join(save_path, 'checkpoint{}.pth'.format(e + 1)))
    print("Finished Training!")


In [14]:
# data info
full_dataset = BraTS21Dataset()
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
# train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(full_dataset,
#                                                                          [train_size, val_size, test_size])
train_dataset = torch.utils.data.Subset(full_dataset, list(range(0, train_size)))
val_dataset = torch.utils.data.Subset(full_dataset, list(range(train_size, train_size + val_size)))
test_dataset = torch.utils.data.Subset(full_dataset, list(range(train_size + val_size, len(full_dataset))))


## Training

In [15]:
# hyperparameters
num_classes = 3
epochs = 60
warmup_epochs = 10
batch_size = 1
lr = 0.0002
min_lr = 0.00002
train_log = '/media/NAS07/USER_PATH/zzx/brats_weight/tucl_300.txt'
weight = '/media/NAS07/USER_PATH/zzx/brats_weight/tucl_300.pth'
save_path = '/media/NAS07/USER_PATH/zzx/brats_weight/tucl_300'

In [17]:
# torch.manual_seed(args.seed)  # 为CPU设置种子用于生成随机数，以使得结果是确定的
# torch.cuda.manual_seed_all(args.seed)  # 为所有的GPU设置种子，以使得结果是确定的

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda:0')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, num_workers=4,  # num_worker=4
                          shuffle=True, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                        pin_memory=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                         pin_memory=True)

print("using {} device.".format(device))
print("using {} images for training, {} images for validation.".format(len(train_dataset), len(val_dataset)))
# img,label = train_dataset[0]

# 1-坏疽(NT,necrotic tumor core),2-浮肿区域(ED,peritumoral edema),4-增强肿瘤区域(ET,enhancing tumor)
# 评价指标：ET(label4), TC(label1+label4), WT(label1+label2+label4)
_, model = TransBTS(dataset='brats', _conv_repr=True, _pe_type="learned")
model.to(device)

criterion = seg_loss
# optimizer = optim.SGD(model.parameters(), momentum=0.9, lr=0, weight_decay=5e-4)
# scheduler = cosine_scheduler(base_value=lr, final_value=min_lr, epochs=epochs,
#                              niter_per_ep=len(train_loader), warmup_epochs=warmup_epochs, start_warmup_value=5e-4)
import torch.optim as optim

# # AdamW optimizer (适用于 Transformer)
# optimizer = optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=5e-4)

# # 余弦调度器 (cosine_scheduler)
# scheduler = cosine_scheduler(
#     base_value=lr, 
#     final_value=min_lr, 
#     epochs=epochs,
#     niter_per_ep=len(train_loader), 
#     warmup_epochs=warmup_epochs, 
#     start_warmup_value=1e-5
# )

optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=5e-4)
scheduler = cosine_scheduler(base_value=lr, final_value=min_lr, epochs=epochs,
                             niter_per_ep=len(train_loader), warmup_epochs=warmup_epochs, start_warmup_value=5e-4)

weight1 = '/media/NAS07/USER_PATH/zzx/brats_weight/tucl_300.pth'
# 加载训练模型
if os.path.exists(weight1):
    weight_dict = torch.load(weight1, map_location=device)
    model.load_state_dict(weight_dict['model'])
    optimizer.load_state_dict(weight_dict['optimizer'])
    print('Successfully loading checkpoint.')

train(model, optimizer, scheduler, criterion, train_loader, val_loader, epochs, device, train_log=train_log)

# metrics1 = val_loop(model, criterion, train_loader, device)
metrics2 = val_loop(model, criterion, val_loader, device)
# metrics3 = test_loop(model, criterion, test_loader, device)

# 最后再测试一遍所有数据，注意，这里使用的是训练结束的模型参数
# print("Train -- loss: {:.3f} ET: {:.3f} TC: {:.3f} WT: {:.3f}".format(metrics1['loss'], metrics1['dice1'],metrics1['dice2'], metrics1['dice3']))
print("Valid -- Dice: ET {:.3f} WT {:.3f} TC {:.3f}"
      .format(
              metrics2['dice1'], metrics2['dice2'], metrics2['dice3']))

using cuda:0 device.
using 210 images for training, 45 images for validation.


You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Successfully loading checkpoint.


loss:0.273 dice1:0.855 dice2:0.789 dice3:0.891 : 100%|█| 210/210 [00:42<00:00,  
dice1:0.637 dice2:0.746 dice3:0.661 : 100%|█████| 45/45 [00:08<00:00,  5.27it/s]


Epoch:[1/60] train_loss: 0.369
Train--ET: 0.756 TC: 0.830 WT: 0.804 
Valid--ET: 0.613 TC: 0.693 WT: 0.642 
save model


...

Epoch:[59/60] train_loss: 0.342
Train--ET: 0.781 TC: 0.850 WT: 0.833 
Valid--ET: 0.564 TC: 0.611 WT: 0.558 


loss:0.403 dice1:0.857 dice2:0.929 dice3:0.844 : 100%|█| 210/210 [00:41<00:00,  
dice1:0.692 dice2:0.766 dice3:0.790 : 100%|█████| 45/45 [00:09<00:00,  4.88it/s]


Epoch:[60/60] train_loss: 0.323
Train--ET: 0.773 TC: 0.851 WT: 0.818 
Valid--ET: 0.644 TC: 0.692 WT: 0.685 
Finished Training!


dice1:0.669 dice2:0.794 dice3:0.751 : 100%|█████| 45/45 [00:09<00:00,  4.78it/s]

Valid -- Dice: ET 0.647 WT 0.692 TC 0.691


In [18]:
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = True
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from scipy import ndimage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                         pin_memory=True)
_, model = TransBTS(dataset='brats', _conv_repr=True, _pe_type="learned")
model.to(device)

criterion = seg_loss
optimizer = optim.SGD(model.parameters(), momentum=0.9, lr=0, weight_decay=5e-4)

# 加载训练模型
if os.path.exists(weight):
    weight_dict = torch.load(weight, map_location=device)
    model.load_state_dict(weight_dict['model'])
    optimizer.load_state_dict(weight_dict['optimizer'])
    print('Successfully loading checkpoint.')

metrics3 = test_loop(model, criterion, test_loader, device, save_dir="/media/NAS07/USER_PATH/zzx/brats_result/Tucl_300")
print("Test  -- Dice: ET {:.3f} WT {:.3f} TC {:.3f} HD95: ET {:.3f} WT {:.3f} TC {:.3f}"
      .format(
              metrics3['dice1'], metrics3['dice2'], metrics3['dice3'],
              metrics3['hd1'], metrics3['hd2'], metrics3['hd3']))

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Successfully loading checkpoint.


the ground truth of class 0 is all 0, this may result in nan/inf distance.7s/it]
the ground truth of class 1 is all 0, this may result in nan/inf distance.
the ground truth of class 2 is all 0, this may result in nan/inf distance.
the prediction of class 0 is all 0, this may result in nan/inf distance..38s/it]
the prediction of class 2 is all 0, this may result in nan/inf distance.
100%|███████████████████████████████████████████| 45/45 [01:07<00:00,  1.51s/it]

Test  -- Dice: ET 0.750 WT 0.739 TC 0.766 HD95: ET nan WT 41.489 TC nan
